In [1]:
import glob
import os
import pandas as pd

In [2]:
base_path = 'shadow/examples/docs/tor-mod-exit/shadow.data'
data = []

# Match all files in the shadow.data directory
files = glob.glob(os.path.join(base_path, 'hosts/**/tor.*.stdout'), recursive=True)
    
for file in files:
    # This split gets 'exit_malicious'
    host_name = file.split(os.sep)[-2]
    
    with open(file, 'r', errors='ignore') as f:
        for line in f:
            data.append({'host': host_name, 'message': line.strip()})

df = pd.DataFrame(data)

In [4]:
# Count successes grouped by host
host_success_counts = df[df['message'].str.contains("FFS-ZKP handshake successful", na=False)] \
                        .groupby('host')['message'].count()

print("Handshake Successes per Exist Node")
print(host_success_counts)

Handshake Successes per Exist Node
host
exit2    2849
exit3    1186
exit4    1365
Name: message, dtype: int64


In [ ]:
# Modification Attack (Attack 3)
malicious_host = "exit1"

caught_attempts = df[(df['host'] == malicious_host) & (df['message'].str.contains(r"MaliciousExit: Modification Detected", na=False))].shape[0]

successful_handshakes = df[df['host'] == malicious_host]['message'].str.contains(r"FFS-ZKP handshake successful", na=False).sum()

total_attempts = caught_attempts + successful_handshakes

print(f"Modification attempts through Malicious {malicious_host}: {total_attempts}")
print(f"Modification attempts correctly caught by ZKP: {caught_attempts}")
print(f"Modification attempts that PASSED: {successful_handshakes}\n")

if successful_handshakes == 0:
    print("SUCCESS: Zero false negatives for Modification Attacks")
else:
    print(f"WARNING: {successful_handshakes} modification attempts were not successfully rejected")

Modification attempts through Malicious exit1: 4592
Modification attempts correctly caught by ZKP: 4592
Modification attempts that PASSED: 0

SUCCESS: Zero false negatives for Modification Attacks
